# Tahap 1 - Membangun Case Base

Project: Case-Based Reasoning untuk Pidana Umum - Pencurian di PN Tangerang

Notebook ini digunakan sebagai bagian dari pipeline CBR.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
BASE_DIR = Path("..").resolve()

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EVAL_DIR = DATA_DIR / "eval"
RESULTS_DIR = DATA_DIR / "results"
LOGS_DIR = BASE_DIR / "logs"

# Create folders if not exist
for folder in [RAW_DIR, PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

BASE_DIR: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang
RAW_DIR: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/raw
PROCESSED_DIR: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed


In [2]:
# Membuat template inventory untuk 40 putusan PN Tangerang - Pidana Umum Pencurian

jumlah_putusan = 40

inventory = pd.DataFrame({
    "case_id": [f"case_{i:03d}" for i in range(1, jumlah_putusan + 1)],
    "no_perkara": ["" for _ in range(jumlah_putusan)],
    "pengadilan": ["PN Tangerang" for _ in range(jumlah_putusan)],
    "jenis_perkara": ["Pidana Umum - Pencurian" for _ in range(jumlah_putusan)],
    "tanggal_putusan": ["" for _ in range(jumlah_putusan)],
    "sumber_url": ["" for _ in range(jumlah_putusan)],
    "raw_file": [f"case_{i:03d}.txt" for i in range(1, jumlah_putusan + 1)],
    "status_download": ["belum" for _ in range(jumlah_putusan)],
    "jumlah_kata": [0 for _ in range(jumlah_putusan)]
})

inventory_path = PROCESSED_DIR / "case_inventory.csv"
inventory.to_csv(inventory_path, index=False)

print("Template inventory berhasil dibuat:")
print(inventory_path)

inventory.head()

Template inventory berhasil dibuat:
/Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed/case_inventory.csv


,case_id,no_perkara,pengadilan,jenis_perkara,tanggal_putusan,sumber_url,raw_file,status_download,jumlah_kata
0,case_001,,PN Tangerang,Pidana Umum - Pencurian,,,case_001.txt,belum,0
1,case_002,,PN Tangerang,Pidana Umum - Pencurian,,,case_002.txt,belum,0
2,case_003,,PN Tangerang,Pidana Umum - Pencurian,,,case_003.txt,belum,0
3,case_004,,PN Tangerang,Pidana Umum - Pencurian,,,case_004.txt,belum,0
4,case_005,,PN Tangerang,Pidana Umum - Pencurian,,,case_005.txt,belum,0


In [6]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()
inventory_path = BASE_DIR / "data" / "processed" / "case_inventory.csv"

# Baca CSV sebagai teks semua agar tidak error saat isi nomor perkara
df = pd.read_csv(inventory_path, dtype=str).fillna("")

# Pastikan kolom penting ada
required_columns = [
    "case_id",
    "no_perkara",
    "pengadilan",
    "jenis_perkara",
    "tanggal_putusan",
    "sumber_url",
    "raw_file",
    "status_download",
    "jumlah_kata"
]

for col in required_columns:
    if col not in df.columns:
        df[col] = ""

# Semua kolom dibuat string dulu supaya aman
for col in required_columns:
    df[col] = df[col].astype(str)

case_id = "case_006"

# Isi data case_001
df.loc[df["case_id"] == case_id, "no_perkara"] = "310/Pid.B/2019/PN.Tng"
df.loc[df["case_id"] == case_id, "tanggal_putusan"] = "18-03-2019"
df.loc[df["case_id"] == case_id, "pengadilan"] = "PN Tangerang"
df.loc[df["case_id"] == case_id, "jenis_perkara"] = "Pidana Umum - Pencurian"
df.loc[df["case_id"] == case_id, "sumber_url"] = "https://putusan3.mahkamahagung.go.id/direktori/putusan/854b1ce2c8150c85537665183232c221.html"
df.loc[df["case_id"] == case_id, "raw_file"] = "case_006.txt"
df.loc[df["case_id"] == case_id, "status_download"] = "belum"
df.loc[df["case_id"] == case_id, "jumlah_kata"] = "0"

# Simpan ulang
df.to_csv(inventory_path, index=False)

print("Data berhasil diperbarui dan disimpan ke:")
print(inventory_path)

df[df["case_id"] == case_id]

Data berhasil diperbarui dan disimpan ke:
/Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed/case_inventory.csv


,case_id,no_perkara,pengadilan,jenis_perkara,tanggal_putusan,sumber_url,raw_file,status_download,jumlah_kata
5,case_006,310/Pid.B/2019/PN.Tng,PN Tangerang,Pidana Umum - Pencurian,18-03-2019,https://putusan3.mahkamahagung.go.id/direktori...,case_006.txt,belum,0


In [11]:
rows_text = """
1790 / PID.B / 2014 / PN.TNG.	14 Oktober 2014	https://putusan3.mahkamahagung.go.id/direktori/putusan/7162a09b85f69cff1006bbeb65358019.html
2118/Pid.B/2014/PN.TNG	18 Desember 2014	https://putusan3.mahkamahagung.go.id/direktori/putusan/a90dfad5ed395a96e45243719e858539.html
1996/Pid.B/2011/PN.TNG	22 Desember 2011	https://putusan3.mahkamahagung.go.id/direktori/putusan/7b8578278a899a37e5346fefeec61638.html
265/PID.B/2013/PN.TNG	21 Maret 2013	https://putusan3.mahkamahagung.go.id/direktori/putusan/1952227d2d0bfff610adcb359e6fe52c.html
1380 / PID.B / 2014 / PN.TNG.	28 Agustus 2014	https://putusan3.mahkamahagung.go.id/direktori/putusan/0af7886969da9821c6e5d1c20163fa05.html
1339/ PID.B/ 2011/ PN.TNG.	22 Agustus 2011	https://putusan3.mahkamahagung.go.id/direktori/putusan/33d09168e3bd3a126ab61880e5aa454a.html
1279/Pid.B/2022/PN Tng	25 Oktober 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed56a5e9aa15deadaa313635303130.html
441/Pid.B/2010/PN.TNG	21 April 2010	https://putusan3.mahkamahagung.go.id/direktori/putusan/1b53a6e54f1d8d2f75b193e9dedcd156.html
917/Pid.B/2019/PN Tng	15 Juli 2019	https://putusan3.mahkamahagung.go.id/direktori/putusan/38755fca77bcfd6d3abbc89817746136.html
1282 / PID.B / 2011 / PN.TNG	10 Agustus 2011	https://putusan3.mahkamahagung.go.id/direktori/putusan/85a681cd7fe2b18180517ffa3351d41e.html
2255/Pid.B/2014/PN.Tng	7 Januari 2015	https://putusan3.mahkamahagung.go.id/direktori/putusan/86bca264b5d325eee8e78dc96460e1a8.html
327/Pid.B/2024/PN Tng	8 Mei 2024	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaef32d1128b9e569b4a313535383231.html
1727/Pid.B/2021/PN Tng	14 Desember 2021	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec5d838cc3615cbc02313534363532.html
784/Pid.B/2022/PN Tng	20 Juli 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed14913d95f1ba95dd313433353535.html
1599/PID.B/2010/PN.TNG	22 Nopember 2010	https://putusan3.mahkamahagung.go.id/direktori/putusan/6551f918f664974a8f47356324632764.html
1973/PID.B/2013/PN.TNG	7 Nopember 2013	https://putusan3.mahkamahagung.go.id/direktori/putusan/efec6384c3624a6ba0312fa37c71a975.html
752/Pid.B/2022/PN Tng	20 Juni 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaecf06be94a1ef4b329313433383030.html
110/PID.B/2012/PN.TNG	23 Februari 2012	https://putusan3.mahkamahagung.go.id/direktori/putusan/db413e1ba15e6bf2febeed1aa93f1e95.html
1644/Pid.B/2021/PN Tng	14 Desember 2021	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec5d83910435d4a3d6313534363539.html
1430/Pid.B/2022/PN Tng	2 Nopember 2022	https://putusan3.mahkamahagung.go.id/direktori/putusan/zaedd2c667f2c0fcbc57313535353130.html
32/Pdt.P/2025/PN Psp		https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf16cab4e3a9c5cb8ef323032343033.html
292/Pid.B/2025/PN Psp		https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf16cab4c54b6fcbc41323032343030.html
272/Pid.B/2025/PN Psp		https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf16cab4a5d78e89838323032333536.html
"""

In [12]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()
inventory_path = BASE_DIR / "data" / "processed" / "case_inventory.csv"

df = pd.read_csv(inventory_path, dtype=str).fillna("")

required_columns = [
    "case_id",
    "no_perkara",
    "pengadilan",
    "jenis_perkara",
    "tanggal_putusan",
    "sumber_url",
    "raw_file",
    "status_download",
    "jumlah_kata"
]

for col in required_columns:
    if col not in df.columns:
        df[col] = ""

for col in required_columns:
    df[col] = df[col].astype(str)

rows = []

for line in rows_text.strip().splitlines():
    parts = line.strip().split("\t")
    
    if len(parts) >= 3:
        no_perkara = parts[0].strip()
        tanggal_putusan = parts[1].strip()
        url = parts[2].strip()
        
        if url.startswith("http") and "/direktori/putusan/" in url:
            rows.append({
                "no_perkara": no_perkara,
                "tanggal_putusan": tanggal_putusan,
                "sumber_url": url
            })

seen = set()
clean_rows = []

for row in rows:
    if row["sumber_url"] not in seen:
        seen.add(row["sumber_url"])
        clean_rows.append(row)

print(f"Jumlah data valid dari browser: {len(clean_rows)}")

existing_urls = set(df["sumber_url"].dropna().astype(str).tolist())

added = 0

for row in clean_rows:
    url = row["sumber_url"]

    if url in existing_urls:
        print(f"Dilewati karena sudah ada: {url}")
        continue

    empty_rows = df.index[(df["sumber_url"] == "") | (df["sumber_url"].isna())].tolist()

    if not empty_rows:
        print("Tidak ada baris kosong lagi.")
        break

    idx = empty_rows[0]
    case_id = df.loc[idx, "case_id"]

    if case_id == "" or case_id.lower() == "nan":
        case_id = f"case_{idx+1:03d}"
        df.loc[idx, "case_id"] = case_id

    df.loc[idx, "no_perkara"] = row["no_perkara"]
    df.loc[idx, "tanggal_putusan"] = row["tanggal_putusan"]
    df.loc[idx, "pengadilan"] = "PN Tangerang"
    df.loc[idx, "jenis_perkara"] = "Pidana Umum - Pencurian"
    df.loc[idx, "sumber_url"] = url
    df.loc[idx, "raw_file"] = f"{case_id}.txt"
    df.loc[idx, "status_download"] = "belum"
    df.loc[idx, "jumlah_kata"] = "0"

    existing_urls.add(url)
    added += 1

df.to_csv(inventory_path, index=False)

print(f"Berhasil menambahkan {added} data baru.")
print("File disimpan ke:", inventory_path)

df[["case_id", "no_perkara", "tanggal_putusan", "sumber_url", "raw_file", "status_download", "jumlah_kata"]].head(40)

Jumlah data valid dari browser: 23
Tidak ada baris kosong lagi.
Berhasil menambahkan 0 data baru.
File disimpan ke: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed/case_inventory.csv


,case_id,no_perkara,tanggal_putusan,sumber_url,raw_file,status_download,jumlah_kata
0,case_001,2184/PID.B/2013/PN.TNG,07-01-2014,https://putusan3.mahkamahagung.go.id/direktori...,case_001.txt,belum,0
1,case_002,1671/Pid.B/2021/PN.Tng,18-10-2021,https://putusan3.mahkamahagung.go.id/direktori...,case_002.txt,belum,0
2,case_003,1885/Pid.B/2022/PN.Tng,10-10-2022,https://putusan3.mahkamahagung.go.id/direktori...,case_003.txt,belum,0
3,case_004,1314/PID.B/2013/PN.TNG,24-07-2013,https://putusan3.mahkamahagung.go.id/direktori...,case_004.txt,belum,0
4,case_005,1493/PID.B/2013/PN.TNG,27-08-2013,https://putusan3.mahkamahagung.go.id/direktori...,case_005.txt,belum,0
5,case_006,310/Pid.B/2019/PN.Tng,18-03-2019,https://putusan3.mahkamahagung.go.id/direktori...,case_006.txt,belum,0
6,case_007,2686/Pid.B/2018/PN Tng,30 Januari 2019,https://putusan3.mahkamahagung.go.id/direktori...,case_007.txt,belum,0
7,case_008,1581/Pid.B/2011/PN.TNG,4 Oktober 2011,https://putusan3.mahkamahagung.go.id/direktori...,case_008.txt,belum,0
8,case_009,1022/Pid.B/2010/PN.TNG,14 Juli 2010,https://putusan3.mahkamahagung.go.id/direktori...,case_009.txt,belum,0
9,case_010,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,https://putusan3.mahkamahagung.go.id/direktori...,case_010.txt,belum,0


In [13]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()
inventory_path = BASE_DIR / "data" / "processed" / "case_inventory.csv"

df = pd.read_csv(inventory_path, dtype=str).fillna("")

jumlah_data = (df["sumber_url"] != "").sum()
jumlah_duplikat = df["sumber_url"].duplicated().sum()

print("Jumlah URL terisi:", jumlah_data)
print("Jumlah URL duplikat:", jumlah_duplikat)

df[["case_id", "no_perkara", "tanggal_putusan", "sumber_url", "raw_file", "status_download", "jumlah_kata"]].head(40)

Jumlah URL terisi: 40
Jumlah URL duplikat: 0


,case_id,no_perkara,tanggal_putusan,sumber_url,raw_file,status_download,jumlah_kata
0,case_001,2184/PID.B/2013/PN.TNG,07-01-2014,https://putusan3.mahkamahagung.go.id/direktori...,case_001.txt,belum,0
1,case_002,1671/Pid.B/2021/PN.Tng,18-10-2021,https://putusan3.mahkamahagung.go.id/direktori...,case_002.txt,belum,0
2,case_003,1885/Pid.B/2022/PN.Tng,10-10-2022,https://putusan3.mahkamahagung.go.id/direktori...,case_003.txt,belum,0
3,case_004,1314/PID.B/2013/PN.TNG,24-07-2013,https://putusan3.mahkamahagung.go.id/direktori...,case_004.txt,belum,0
4,case_005,1493/PID.B/2013/PN.TNG,27-08-2013,https://putusan3.mahkamahagung.go.id/direktori...,case_005.txt,belum,0
5,case_006,310/Pid.B/2019/PN.Tng,18-03-2019,https://putusan3.mahkamahagung.go.id/direktori...,case_006.txt,belum,0
6,case_007,2686/Pid.B/2018/PN Tng,30 Januari 2019,https://putusan3.mahkamahagung.go.id/direktori...,case_007.txt,belum,0
7,case_008,1581/Pid.B/2011/PN.TNG,4 Oktober 2011,https://putusan3.mahkamahagung.go.id/direktori...,case_008.txt,belum,0
8,case_009,1022/Pid.B/2010/PN.TNG,14 Juli 2010,https://putusan3.mahkamahagung.go.id/direktori...,case_009.txt,belum,0
9,case_010,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,https://putusan3.mahkamahagung.go.id/direktori...,case_010.txt,belum,0


In [14]:
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import time

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
LOGS_DIR = BASE_DIR / "logs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

inventory_path = PROCESSED_DIR / "case_inventory.csv"
log_path = LOGS_DIR / "download.log"

df = pd.read_csv(inventory_path, dtype=str).fillna("")

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://putusan3.mahkamahagung.go.id/"
})


def clean_text(text):
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\t", " ", text)
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ ]+", " ", text)
    text = text.strip()
    return text


def html_to_text(html):
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = clean_text(text)
    return text


def fetch_text_from_url(url):
    response = session.get(url, timeout=30)

    if response.status_code != 200:
        return "", f"gagal_{response.status_code}"

    html = response.text
    text = html_to_text(html)

    if len(text.split()) < 50:
        return text, "berhasil_tapi_teks_pendek"

    return text, "berhasil"


logs = []

for idx, row in df.iterrows():
    case_id = row.get("case_id", f"case_{idx+1:03d}")
    url = row.get("sumber_url", "")
    raw_file = row.get("raw_file", f"{case_id}.txt")

    if not url:
        df.loc[idx, "status_download"] = "url_kosong"
        continue

    output_path = RAW_DIR / raw_file

    print(f"Memproses {case_id}: {url}")

    try:
        text, status = fetch_text_from_url(url)
        jumlah_kata = len(text.split())

        if text:
            output_path.write_text(text, encoding="utf-8")

        df.loc[idx, "status_download"] = status
        df.loc[idx, "jumlah_kata"] = str(jumlah_kata)

        logs.append(f"{case_id} | {status} | {jumlah_kata} kata | {url}")

    except Exception as e:
        df.loc[idx, "status_download"] = "gagal_error"
        df.loc[idx, "jumlah_kata"] = "0"

        logs.append(f"{case_id} | gagal_error | {str(e)} | {url}")

    time.sleep(1)

df.to_csv(inventory_path, index=False)

log_path.write_text("\n".join(logs), encoding="utf-8")

print("\nProses selesai.")
print("Inventory diperbarui:", inventory_path)
print("Log tersimpan:", log_path)

df[["case_id", "status_download", "jumlah_kata", "raw_file"]].head(40)

Memproses case_001: https://putusan3.mahkamahagung.go.id/direktori/putusan/920f583c6a4c4c1e857abd142bcaae4c.html
Memproses case_002: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec79c6b87c3daa8982313435383134.html
Memproses case_003: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed970b08fe45feb40e313533353136.html
Memproses case_004: https://putusan3.mahkamahagung.go.id/direktori/putusan/bd7607df513ced610ce107010c47cc1b.html
Memproses case_005: https://putusan3.mahkamahagung.go.id/direktori/putusan/adba62392c732ab4baa77debec9bbd72.html
Memproses case_006: https://putusan3.mahkamahagung.go.id/direktori/putusan/854b1ce2c8150c85537665183232c221.html
Memproses case_007: https://putusan3.mahkamahagung.go.id/direktori/putusan/ae04fe178665082c029a9b8bab0f2e4a.html
Memproses case_008: https://putusan3.mahkamahagung.go.id/direktori/putusan/25a935db4781ac365604657c7a16d549.html
Memproses case_009: https://putusan3.mahkamahagung.go.id/direktori/putusan/02d8373b52a74b64cb2b5

,case_id,status_download,jumlah_kata,raw_file
0,case_001,gagal_403,0,case_001.txt
1,case_002,gagal_403,0,case_002.txt
2,case_003,gagal_403,0,case_003.txt
3,case_004,gagal_403,0,case_004.txt
4,case_005,gagal_403,0,case_005.txt
5,case_006,gagal_403,0,case_006.txt
6,case_007,gagal_403,0,case_007.txt
7,case_008,gagal_403,0,case_008.txt
8,case_009,gagal_403,0,case_009.txt
9,case_010,gagal_403,0,case_010.txt


In [1]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 179.9 kB/s eta 0:00:00m eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 221.7 kB/s eta 0:00:001m213.4 kB/s eta 0:00:01


In [16]:
from pathlib import Path
import pandas as pd
import time
import re

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
LOGS_DIR = BASE_DIR / "logs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

inventory_path = PROCESSED_DIR / "case_inventory.csv"
log_path = LOGS_DIR / "selenium_download.log"

df = pd.read_csv(inventory_path, dtype=str).fillna("")


def clean_text(text):
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\t", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    return text.strip()


def extract_no_perkara_from_text(text):
    patterns = [
        r"Nomor\s*[:\-]?\s*([0-9]+\/Pid\.B\/[0-9]{4}\/PN\.?\s*Tng)",
        r"Nomor\s*[:\-]?\s*([0-9]+\/PID\.B\/[0-9]{4}\/PN\.?\s*TNG)",
        r"([0-9]+\/Pid\.B\/[0-9]{4}\/PN\.?\s*Tng)",
        r"([0-9]+\/PID\.B\/[0-9]{4}\/PN\.?\s*TNG)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).replace(" ", "")
    
    return ""


def extract_tanggal_from_text(text):
    bulan = "Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember"

    patterns = [
        rf"tanggal\s+([0-9]{{1,2}}\s+(?:{bulan})\s+[0-9]{{4}})",
        rf"putus\s+[:\-]?\s*([0-9]{{1,2}}\s+(?:{bulan})\s+[0-9]{{4}})",
        rf"([0-9]{{1,2}}\s+(?:{bulan})\s+[0-9]{{4}})",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1)
    
    return ""


# Setting Chrome
chrome_options = Options()

# Jangan headless dulu supaya seperti browser biasa
# Chrome akan terbuka otomatis
chrome_options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=chrome_options)

logs = []

try:
    for idx, row in df.iterrows():
        case_id = row.get("case_id", f"case_{idx+1:03d}")
        url = row.get("sumber_url", "")
        raw_file = row.get("raw_file", f"{case_id}.txt")

        if not url:
            df.loc[idx, "status_download"] = "url_kosong"
            df.loc[idx, "jumlah_kata"] = "0"
            continue

        output_path = RAW_DIR / raw_file

        print(f"\nMemproses {case_id}")
        print(url)

        try:
            driver.get(url)

            # beri waktu halaman terbuka
            time.sleep(4)

            body = driver.find_element(By.TAG_NAME, "body")
            text = body.text
            text = clean_text(text)

            jumlah_kata = len(text.split())

            if "403" in text[:300] or "forbidden" in text[:300].lower():
                status = "gagal_browser_403"
                jumlah_kata = 0
            elif jumlah_kata < 50:
                status = "berhasil_tapi_teks_pendek"
            else:
                status = "berhasil"

            if jumlah_kata > 0:
                output_path.write_text(text, encoding="utf-8")

            # Update metadata jika masih kosong
            if df.loc[idx, "no_perkara"] == "":
                no_perkara = extract_no_perkara_from_text(text)
                df.loc[idx, "no_perkara"] = no_perkara

            if df.loc[idx, "tanggal_putusan"] == "":
                tanggal_putusan = extract_tanggal_from_text(text)
                df.loc[idx, "tanggal_putusan"] = tanggal_putusan

            df.loc[idx, "status_download"] = status
            df.loc[idx, "jumlah_kata"] = str(jumlah_kata)
            df.loc[idx, "raw_file"] = raw_file

            logs.append(f"{case_id} | {status} | {jumlah_kata} kata | {url}")

            print(f"Status: {status}")
            print(f"Jumlah kata: {jumlah_kata}")

            # Simpan progress setiap 1 kasus agar aman
            df.to_csv(inventory_path, index=False)
            log_path.write_text("\n".join(logs), encoding="utf-8")

        except Exception as e:
            df.loc[idx, "status_download"] = "gagal_error"
            df.loc[idx, "jumlah_kata"] = "0"

            logs.append(f"{case_id} | gagal_error | {str(e)} | {url}")

            print("Gagal:", e)

            df.to_csv(inventory_path, index=False)
            log_path.write_text("\n".join(logs), encoding="utf-8")

        # jeda agar tidak terlalu cepat
        time.sleep(3)

finally:
    driver.quit()

df.to_csv(inventory_path, index=False)
log_path.write_text("\n".join(logs), encoding="utf-8")

print("\nProses Selenium selesai.")
print("Inventory diperbarui:", inventory_path)
print("Log tersimpan:", log_path)

df[["case_id", "no_perkara", "tanggal_putusan", "status_download", "jumlah_kata", "raw_file"]].head(40)


Memproses case_001
https://putusan3.mahkamahagung.go.id/direktori/putusan/920f583c6a4c4c1e857abd142bcaae4c.html
Status: berhasil_tapi_teks_pendek
Jumlah kata: 33

Memproses case_002
https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec79c6b87c3daa8982313435383134.html
Status: berhasil_tapi_teks_pendek
Jumlah kata: 41

Memproses case_003
https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed970b08fe45feb40e313533353136.html
Status: berhasil_tapi_teks_pendek
Jumlah kata: 33

Memproses case_004
https://putusan3.mahkamahagung.go.id/direktori/putusan/bd7607df513ced610ce107010c47cc1b.html
Status: berhasil_tapi_teks_pendek
Jumlah kata: 41

Memproses case_005
https://putusan3.mahkamahagung.go.id/direktori/putusan/adba62392c732ab4baa77debec9bbd72.html
Status: berhasil_tapi_teks_pendek
Jumlah kata: 41

Memproses case_006
https://putusan3.mahkamahagung.go.id/direktori/putusan/854b1ce2c8150c85537665183232c221.html
Status: berhasil_tapi_teks_pendek
Jumlah kata: 41

Memproses case_007
ht

,case_id,no_perkara,tanggal_putusan,status_download,jumlah_kata,raw_file
0,case_001,2184/PID.B/2013/PN.TNG,07-01-2014,berhasil_tapi_teks_pendek,33,case_001.txt
1,case_002,1671/Pid.B/2021/PN.Tng,18-10-2021,berhasil_tapi_teks_pendek,41,case_002.txt
2,case_003,1885/Pid.B/2022/PN.Tng,10-10-2022,berhasil_tapi_teks_pendek,33,case_003.txt
3,case_004,1314/PID.B/2013/PN.TNG,24-07-2013,berhasil_tapi_teks_pendek,41,case_004.txt
4,case_005,1493/PID.B/2013/PN.TNG,27-08-2013,berhasil_tapi_teks_pendek,41,case_005.txt
5,case_006,310/Pid.B/2019/PN.Tng,18-03-2019,berhasil_tapi_teks_pendek,41,case_006.txt
6,case_007,2686/Pid.B/2018/PN Tng,30 Januari 2019,berhasil_tapi_teks_pendek,41,case_007.txt
7,case_008,1581/Pid.B/2011/PN.TNG,4 Oktober 2011,berhasil_tapi_teks_pendek,41,case_008.txt
8,case_009,1022/Pid.B/2010/PN.TNG,14 Juli 2010,berhasil_tapi_teks_pendek,41,case_009.txt
9,case_010,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,berhasil_tapi_teks_pendek,41,case_010.txt


In [17]:
df = pd.read_csv(inventory_path, dtype=str).fillna("")

print(df["status_download"].value_counts())

df[["case_id", "no_perkara", "tanggal_putusan", "status_download", "jumlah_kata", "raw_file"]].head(40)

status_download
berhasil_tapi_teks_pendek    40
Name: count, dtype: int64


,case_id,no_perkara,tanggal_putusan,status_download,jumlah_kata,raw_file
0,case_001,2184/PID.B/2013/PN.TNG,07-01-2014,berhasil_tapi_teks_pendek,33,case_001.txt
1,case_002,1671/Pid.B/2021/PN.Tng,18-10-2021,berhasil_tapi_teks_pendek,41,case_002.txt
2,case_003,1885/Pid.B/2022/PN.Tng,10-10-2022,berhasil_tapi_teks_pendek,33,case_003.txt
3,case_004,1314/PID.B/2013/PN.TNG,24-07-2013,berhasil_tapi_teks_pendek,41,case_004.txt
4,case_005,1493/PID.B/2013/PN.TNG,27-08-2013,berhasil_tapi_teks_pendek,41,case_005.txt
5,case_006,310/Pid.B/2019/PN.Tng,18-03-2019,berhasil_tapi_teks_pendek,41,case_006.txt
6,case_007,2686/Pid.B/2018/PN Tng,30 Januari 2019,berhasil_tapi_teks_pendek,41,case_007.txt
7,case_008,1581/Pid.B/2011/PN.TNG,4 Oktober 2011,berhasil_tapi_teks_pendek,41,case_008.txt
8,case_009,1022/Pid.B/2010/PN.TNG,14 Juli 2010,berhasil_tapi_teks_pendek,41,case_009.txt
9,case_010,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,berhasil_tapi_teks_pendek,41,case_010.txt


In [18]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

inventory_path = PROCESSED_DIR / "case_inventory.csv"

df = pd.read_csv(inventory_path, dtype=str).fillna("")

def is_cloudflare_text(text):
    markers = [
        "cloudflare",
        "melakukan verifikasi keamanan",
        "ray id",
        "bot jahat",
        "performa dan keamanan",
        "privasi"
    ]
    text_lower = text.lower()
    return any(marker in text_lower for marker in markers)

reset_count = 0

for idx, row in df.iterrows():
    raw_file = row.get("raw_file", "")
    raw_path = RAW_DIR / raw_file

    if raw_path.exists():
        text = raw_path.read_text(encoding="utf-8", errors="ignore")
        
        if is_cloudflare_text(text):
            raw_path.unlink()
            df.loc[idx, "status_download"] = "gagal_cloudflare"
            df.loc[idx, "jumlah_kata"] = "0"
            reset_count += 1

df.to_csv(inventory_path, index=False)

print("Jumlah file Cloudflare yang dihapus/reset:", reset_count)

df[["case_id", "status_download", "jumlah_kata", "raw_file"]].head(40)

Jumlah file Cloudflare yang dihapus/reset: 40


,case_id,status_download,jumlah_kata,raw_file
0,case_001,gagal_cloudflare,0,case_001.txt
1,case_002,gagal_cloudflare,0,case_002.txt
2,case_003,gagal_cloudflare,0,case_003.txt
3,case_004,gagal_cloudflare,0,case_004.txt
4,case_005,gagal_cloudflare,0,case_005.txt
5,case_006,gagal_cloudflare,0,case_006.txt
6,case_007,gagal_cloudflare,0,case_007.txt
7,case_008,gagal_cloudflare,0,case_008.txt
8,case_009,gagal_cloudflare,0,case_009.txt
9,case_010,gagal_cloudflare,0,case_010.txt


In [7]:
from pathlib import Path
import pandas as pd
import json

BASE_DIR = Path("..").resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"

inventory_path = PROCESSED_DIR / "case_inventory.csv"
script_path = BASE_DIR / "browser_extract_script.js"

df = pd.read_csv(inventory_path, dtype=str).fillna("")

rows = df[df["sumber_url"] != ""][
    ["case_id", "no_perkara", "tanggal_putusan", "sumber_url", "raw_file"]
].copy()

rows = rows.rename(columns={"sumber_url": "url"})

records = rows.to_dict(orient="records")

js_code = f"""
(async () => {{
  const rows = {json.dumps(records, ensure_ascii=False, indent=2)};

  const sleep = (ms) => new Promise(resolve => setTimeout(resolve, ms));

  const cleanText = (text) => {{
    return text
      .replace(/\\r/g, "\\n")
      .replace(/\\t/g, " ")
      .replace(/\\n{{3,}}/g, "\\n\\n")
      .replace(/[ ]{{2,}}/g, " ")
      .trim();
  }};

  const isSecurityPage = (text) => {{
    const lower = text.toLowerCase();
    return (
      lower.includes("cloudflare") ||
      lower.includes("melakukan verifikasi keamanan") ||
      lower.includes("ray id") ||
      lower.includes("bot jahat") ||
      lower.includes("performa dan keamanan")
    );
  }};

  const results = [];

  console.log("Mulai mengambil teks dari", rows.length, "URL");

  for (let i = 0; i < rows.length; i++) {{
    const row = rows[i];

    console.log(`[${{i + 1}}/${{rows.length}}] Memproses ${{row.case_id}}: ${{row.url}}`);

    try {{
      const response = await fetch(row.url, {{
        credentials: "include",
        cache: "no-store"
      }});

      const html = await response.text();

      const doc = new DOMParser().parseFromString(html, "text/html");

      doc.querySelectorAll("script, style, nav, footer, header").forEach(el => el.remove());

      let text = "";

      if (doc.body) {{
        text = doc.body.innerText || doc.body.textContent || "";
      }}

      text = cleanText(text);

      const wordCount = text.length > 0 ? text.split(/\\s+/).filter(Boolean).length : 0;

      let status = "berhasil_html";

      if (response.status !== 200) {{
        status = "gagal_http_" + response.status;
      }} else if (isSecurityPage(text)) {{
        status = "gagal_cloudflare";
      }} else if (wordCount < 50) {{
        status = "teks_pendek";
      }}

      results.push({{
        case_id: row.case_id,
        no_perkara: row.no_perkara,
        tanggal_putusan: row.tanggal_putusan,
        raw_file: row.raw_file,
        url: row.url,
        status: status,
        http_status: response.status,
        jumlah_kata: wordCount,
        text: text
      }});

      console.log(row.case_id, status, wordCount, "kata");

    }} catch (err) {{
      results.push({{
        case_id: row.case_id,
        no_perkara: row.no_perkara,
        tanggal_putusan: row.tanggal_putusan,
        raw_file: row.raw_file,
        url: row.url,
        status: "gagal_error",
        error: String(err),
        jumlah_kata: 0,
        text: ""
      }});

      console.error("Gagal:", row.case_id, err);
    }}

    await sleep(1500);
  }}

  const blob = new Blob([JSON.stringify(results, null, 2)], {{
    type: "application/json"
  }});

  const a = document.createElement("a");
  a.href = URL.createObjectURL(blob);
  a.download = "ma_putusan_texts.json";
  document.body.appendChild(a);
  a.click();
  document.body.removeChild(a);

  console.log("Selesai. File ma_putusan_texts.json sudah dibuat/download.");
}})();
"""

script_path.write_text(js_code, encoding="utf-8")

print("Script browser berhasil dibuat:")
print(script_path)
print()
print("Buka file browser_extract_script.js, copy semua isinya, lalu paste ke Console Chrome di halaman MA.")

Script browser berhasil dibuat:
/home/zack/Penalaran-Komputer-subcpmk-3/browser_extract_script.js

Buka file browser_extract_script.js, copy semua isinya, lalu paste ke Console Chrome di halaman MA.


In [8]:
page1_json = r"""
[
  {
    "no_perkara": "1022/Pid.B/2010/PN.TNG",
    "tanggal_putusan": "14 Juli 2010",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/02d8373b52a74b64cb2b509e597ab22e.html",
    "title": "Putusan PN TANGERANG Nomor 1022/Pid.B/2010/PN.TNG",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Kejahatan terhadap keamanan negara Putus : 14-07-2010 — Upload : 16-07-2012 Putusan PN TANGERANG Nomor 1022/Pid.B/2010/PN.TNG Tanggal 14 Juli 2010 — JAMALUDIN Bin SANIF 23 — 1 Menyatakan Terdakwa JAMALUDIN Bin SANIF terbukti secara sah danmeyakinkan bersalah melakukan tindak pidana pencurian dalam, sebagaimana diatur dalam Pasal 363 ayat (2) KUHP; 2. Menjatuhkan pidana terhadap terdakwa JAMALUDIN Bin SANIF denganpidana penjara selama 1 (satu) tahun dikurangi selama terdakwaberada dalam tahanan sementara dengan perintah terdakwa tetapditahan ;3."
  },
  {
    "no_perkara": "497 / PID.B / 2014 / PN.TNG.",
    "tanggal_putusan": "19 Mei 2014",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/8759143bae7fc4503fe79b069b2b8b1b.html",
    "title": "Putusan PN TANGERANG Nomor 497 / PID.B / 2014 / PN.TNG.",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Pencurian Putus : 19-05-2014 — Upload : 22-09-2014 Putusan PN TANGERANG Nomor 497 / PID.B / 2014 / PN.TNG. Tanggal 19 Mei 2014 — AMIN Als UBE Bin UDIN dan ALIP KURNIAWAN Als ONYIP Bin MADIN 34 — 5 ONYIP Bin MADIN telah bersalah melakukan tindak pidanasecara bersamasama melakuakn pencurian dengan kekerasan sebagaimanadiatur dan diancam dalam Pasal 365 ayat (2) ke2 KUHP sesuai dengan dakwaanJaksa Penuntut Umum ;Menjatuhkan pidana penjara terhadap Terdakwa AMIN Als. UBE Bin UDIN danALIP KURNIAWAN Als. BNP2 TKI Kelurahan Selapajang JayaKecamatan Neglasari Kota Tangerang atau setidaktidaknya di suatu tempat lain yangmasih termasuk dalam daerah hukum Pengadilan Negeri Tangerang yang berwenangmemeriksa dan mengadili perkara tersebut, mengambil barang sesuatu, yang seluruhnyaatau sebagian kepunyaan orang lain dengan maksud untuk dimiliki secara melawanhukum, yang didahului, disertai atau diikuti dengan kekerasan atau ancaman kekerasan,terhadap orang dengan maksud untuk mempersiapkan atau mempemrudah pencurian Saksi ELFAN ADITYA PRAYOGA BIN DADANG EFENDI:e Bahwa saksi pernah diperiksa di Polisi ;e Bahwa keterangan yang pernah saksi terangkan di Polisi tersebut benar ;e Bahwa pada hari Minggu tanggal 01 Desember 2013 sekira pukul 17.30 Wibdi Jalan BNP2TKI Kelurahan Selapajang Jaya Kecamatan Neglasari KotaTangerang, Para Terdakwa dan saksi HARIS ALIAS BULUK BIN MURSINtelah melakukan pencurian terhadap barang berupa 1 (satu) unit HP MerkNokia Type C3 warna abuabu milik saksi ;2. Saksi HARIS ALIAS BULUK BIN MURSIN :Bahwa saksi pernah diperiksa di Polisi ;Bahwa keterangan yang pernah saksi terangkan di Polisi tersebut benar ;Bahwa pada hari Minggu tanggal 01 Desember 2013 sekira pukul 17.30 Wibdi Jalan BNP2TKI Kelurahan Selapajang Jaya Kecamatan Neglasari KotaTangerang, saya dan Para Terdakwa telah melakukan pencurian terhadapbarang berupa (satu) unit HP Merk Nokia Type C3 warna abuabu miliksaksi ELFAN ADITYA PRAYOGA BIN DADANG EFENDI ;Menimbang, bahwa atas keterangan saksisaksi UBE Bin UDIN dan Terdakwa II.ALIP KURNIAWAN Als ONYIP Bin MADIN terbukti secara sah danmeyakinkan bersalah melakukan tindak pidana Pencurian dengankekerasan ;Menjatuhkan pidana terhadap Terdakwa I. AMIN Als. UBE Bin UDIN danTerdakwa II."
  },
  {
    "no_perkara": "1885 /Pid.B/2011/PN.TNG",
    "tanggal_putusan": "15 Desember 2011",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/6c202caab5f21bbfbb02c645b1e120b1.html",
    "title": "Putusan PN TANGERANG Nomor 1885 /Pid.B/2011/PN.TNG",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Kejahatan terhadap keamanan negara Putus : 15-12-2011 — Upload : 14-05-2012 Putusan PN TANGERANG Nomor 1885 /Pid.B/2011/PN.TNG Tanggal 15 Desember 2011 — MAULANA HASANUDIN als. KEDOK bin ROJALI 37 — 2 KEDOK binROJALI bersalah melakukan tindak pidana Pencurian denganpemberatan dalam surat dakwaan kami ;2 Menjatuhkan pidana penjara' terhadap Terdakwa MAULANAHASANUDIN als. KEDOK binROJALI telah terbukti secara sah dan meyakinkan bersalah melakukan tindakpidana Pencurian dalam keadaan memberatkan ;2 Menjatuhkan pidana terhadap Terdakwa oleh karena itu dengan pidana penjaraselama 8 (delapan) bulan ;3 Menetapkan masa penangkapan dan atau penahanan yang telah dijalankan olehTerdakwa dikurangkan seluruhnya dari pidana yang dijatuhkan ;4 Memerintahkan agar Terdakwa tetap berada dalam tahanan ;5 Menetapkan barang bukti berupa : nihil ;6 Membebankan kepada Terdakwa untuk membayar"
  },
  {
    "no_perkara": "1073/Pid.B/2019/PN Tng",
    "tanggal_putusan": "10 Juli 2019",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/3ef0dd933715a8193d2f3eb06a2f5cf1.html",
    "title": "Putusan PN TANGERANG Nomor 1073/Pid.B/2019/PN Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Register : 17-05-2019 — Putus : 10-07-2019 — Upload : 24-07-2019 Putusan PN TANGERANG Nomor 1073/Pid.B/2019/PN Tng Tanggal 10 Juli 2019 — Penuntut Umum: REZA VAHLEVI, SH Terdakwa: MUHAMMAD RIKI YAKUB Alias INYONG Bin RIPIN 37 — 3 Menyatakan Terdakwa MUHAMMAD RIKI YAKUB Alias INYONG BinRIPIN terbukti secara sah dan meyakinkan bersalah melakukan tindakpidana pencurian sebagaimana dalam dakwaan Pasai 362 KUHP2. Menjatuhkan pidana terhadap Terdakwa MUHAMMAD RIKI YAKUBAlias INYONG Bin RIPIN dengan pidana penjara seiama 3 (tiga) Bulandikurangi selama terdakwa berada dalam tahanan dengan perintah agartetap ditahan.3. sementara motor tersebutTerdakwa MUHAMMAD RIKI YAKUB Alias INYONG Bin RIPIN gadaikankepada Sdr.NAWIR melalui Sdr.HADI Alias DOYOK sebesar Rp.3.000.000,(tiga juta rupiah) yang mana uangnya telah habis digunakan untukkebutuhan seharihari.Menimbang, bahwa unsurunsur ini telah terbukti secara sah danmeyakinkan atas diri Terdakwa;Menimbang, bahwa oleh karena semua unsur dari Pasal 362 KUHPtelah terpenuhi, maka Terdakwa haruslah dinyatakan telah terbukti secara sahdan meyakinkan melakukan tindak pidana Pencurian Menyatakan Terdakwa Muhammad Riki Yakub Alias Inyong Bin Ripintelah terbukti secara sah dan meyakinkan bersalah melakukan tindakpidana Pencurian;2. Menjatuhkan pidana oleh karena itu terhadap Terdakwa MuhammadRiki Yakub Alias Inyong Bin Ripin tersebut dengan pidana penjaraselama 4 (empat) bulan;3. Menetapkan masa penangkapan dan penahanan yang telah dijalaniterdakwa dikurangkan seluruhnya dari pidana yang dijatuhkan;4. Menetapkan agar terdakwa tetap berada dalam tahanan;5."
  },
  {
    "no_perkara": "678/ PID.B/ 2011/ PN TNG",
    "tanggal_putusan": "8 Mei 2012",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/18665f313e4f85241b5592b8d5212fa2.html",
    "title": "Putusan PN TANGERANG Nomor 678/ PID.B/ 2011/ PN TNG",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Pencurian Putus : 08-05-2012 — Upload : 22-02-2015 Putusan PN TANGERANG Nomor 678/ PID.B/ 2011/ PN TNG Tanggal 8 Mei 2012 — MOHAMAD HERI Als HERI Bin SAMIR 35 — 10 /2011/PN.TNG. tentang penetapan hari sidang perkara ini ;Telah membaca semua suratsurat dalam perkara yang bersangkutan ;Telah mendengar pembacaan Surat Dakwaan Penuntut Umum di mukapersidangan ;Telah mendengar keterangan saksisaksi dan Terdakwa di persidangan ;Telah mendengar pembacaan tuntutan pidana Penuntut Umum pada KejaksaanNegeri Tigaraksa, yang disampaikan di persidangan pada hari Kamis tanggal 16 Juni2011, yang berkesimpulan bahwa Terdakwa telah terbukti bersalah melakukan tindakpidana pencurian Menyatakan terdakwa MOHAMAD HERI Als HERI Bin SAMIR telah terbukti secarasah dan meyakinkan melakukan Tindal< Pidana \"Pencurian\" sebagaimana diatur clandiancam Pidana dalam Pasal 362 KUHP dalam dakwaan kami ;1 Menjatuhkan pidana terhadap terdakwa MOHAMAD HERI Als HERI Bin SAMIRdengan pidana penjara selama 8 (delapan) bulan Penjara dikurangi selama terdakwaberada dalam tahanan sementara dengan perintah agar terdakwa tetap ditahan ;2 Menyatakan Barang bukti berupa:e 1 (satu) potong celana kolor warna Tangerang telah bekerja ditoko tersebut selama 15 (lima betas) tahun ;Bahwa pada hari Sabtu tanggal 22 Oktoher 2011 sekitar pukul 13.00 WI diToko Alan ternpat saksi SUWITA bekerja telah terjadi pencurian bent pa 2(dua) bat rokok Djisemsoe yang tclob dipesan oleh Sdr. SUWITA bersama Saksi AFAN dan koryawan toko Alanlainnya melihat hesil rekarnan CCTV toko Afan clan diketahui dari nisirekamen CCTV tersebut hi hwe pefeku pencurian yang telah mengambil 2(dua) bal rokok Djisamsoe yang telah dipesan oleh Sdr. DONO adalahterdakwa MOHAMAD HERI Als. HERI Bin SAMIR ;e Bahwa pada saat Sdr. DONO datang ke toko Alan dan memesan rokoktersebut saksi SU WIT A tidek mengetahuinve den a pa ka h Sdr. DON 0detang horse ma terdakwa MOHAMAD HERI Als. Tangerang den telah bekerja ditoko tersebut selama 5 (lima) tahun ;Bahwa pada hari Sabtu tanggal 22 Oktober 2011 sekitar pukul 3.00 di TokoAfan tempat saksi ERNAWATI bekerja telah terjadi pencurian berupa 2 (due)bal rokok Djisarnsoe yang tela.h. dipesan oleh Sdr. DONO, yang pelakunya saksiERNAWATI ketahui dari rekeman CCTV toko Alan adalah terdakwa MC)II AMA'DII,RI Als. Bin."
  },
  {
    "no_perkara": "1527/Pid.B/2014/PN.TNG",
    "tanggal_putusan": "16 September 2014",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/a1494e35aeb849e79506ae4ca8c28be2.html",
    "title": "Putusan PN TANGERANG Nomor 1527/Pid.B/2014/PN.TNG",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Pencurian Putus : 16-09-2014 — Upload : 01-03-2015 Putusan PN TANGERANG Nomor 1527/Pid.B/2014/PN.TNG Tanggal 16 September 2014 — NUR APRIYANI Binti (Alm) NURDIN 28 — 9 B/2014/PN.TNG.Setelah mendengar pembacaan tuntutan pidana yang diajukan oleh PenuntutUmum yang pada pokoknya sebagai berikut :1 Menyatakan Terdakwa NUR APRIYANI Binti (Alm) NURDIN, terbuktibersalah melakukan tindak pidana Pencurian, sebagaimana diatur dandiancam pidana hukuman berdasarkan ketentuan Pasal 362 KUHP;2 Pidana penjara terhadap Terdakwa selama 6 (enam) bulan, dikurangi selamaTerdakwa berada dalam tahanan;3 Menetapkan barang bukti berupa :e 1 (satu) buah tas wanita merk PRADA warna merah sudah diamankan, atas kejadiantersebut pelaku berikut barang bukti diamankan ke Polsek Serpong gunapenyelidikan lebih lanjut;Bahwa atas keterangan saksi tersebut Terdakwa tidak keberatan danmembenarkannya;Menimbang, bahwa Terdakwa NUR APRIYANI Binti (Alm) NURDIN dipersidangan telah memberikan keterangan yang pada pokoknya sebagai berikut:e Bahwa Terdakwa dalam keadaan sehat jasmani dan rohani dan telahmemberikan keterangan pada Penyidik di Kepolisian dan membenarkannya;e Bahwa Terdakwa melakukan pencurian Mengambil barang sesuatu;Yang sebagian atau seluruhnya kepunyaan orang Iain;Dengan maksud untuk dimiliki;mM KR & NO Secara melawan hukum;Menimbang, bahwa berdasarkan keterangan saksisaksi, keterangan terdakwa,petunjuk surat dan adanya barang bukti serta faktafakta yang terungkap dalampersidangan, Majelis Hakim sependapat dengan Penuntut Umum bahwa seluruh unsurunsur Pasal 362 KUHP telah terpenuhi, dan oleh karenanya Terdakwa telah terbuktisecara sah dan meyakinkan bersalah melakukan tindak pidana Pencurian Terdakwa sopan dalam persidangan;e Terdakwa mengaku terus terang perbuatannya;e Terdakwa belum pernah dihukum;Menimbang, bahwa oleh karena Terdakwa dijatuhi pidana, maka haruslahdibebani pula untuk membayar biaya perkara;Memperhatikan, Pasal 362 KUHP dan Undangundang Nomor 8 Tahun 1981tentang Hukum Acara Pidana serta peraturan perundangundangan lain yangbersangkutan;MENGADILI:1 Menyatakan Terdakwa NUR APRIYANI Binti (Alm) NURDIN, terbuktisecara sah dan meyakinkan bersalah melakukan tindak pidana Pencurian"
  },
  {
    "no_perkara": "834 / Pid.B / 2017 / PN.TNG.",
    "tanggal_putusan": "15 Juni 2017",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/f00d054e938e4c0c46bc04ca0cdc973d.html",
    "title": "Putusan PN TANGERANG Nomor 834 / Pid.B / 2017 / PN.TNG.",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Pencurian Putus : 15-06-2017 — Upload : 27-12-2017 Putusan PN TANGERANG Nomor 834 / Pid.B / 2017 / PN.TNG. Tanggal 15 Juni 2017 — DEDY KUMALA BIN SAHLAN 60 — 4 Menyatakan Terdakwa DEDY KUMALA BIN SAHLAN bersalah secara syahdan meyakinkan melakukan tindak pidana Pencurian sebagaimana dalamdakwaan Pasal 362 KUHP ;2. Menjatuhkan pidana penjara kepada Terdakwa DEDY KUMALA BINSAHLAN selama 1 (satu) tahun dan 6 (enam) bulan dikurangi masapenahanan yang telah dijalankan Terdakwa dengan perintah agar terdakwatetap ditahan ;.3. Menyatakan barang bukti berupa : 1 (satu) lembar STNK sepeda motor merk Yamaha Jupiter MX No. Pol B6427NWP ; Bahwa benar yang mengambil sepeda motor saksi adalah Terdakwa ; Bahwa sebelum kejadian pencurian, sepeda motor milik saksi diparkir didepan rumah ; Bahwa saksi tahu, ada pencurian setelah sepeda motor saksi jatuh danterdakwa kabur dan akhirnya tertangkap ;2. SALIMparkir sepeda motor dipinggir jalan depan rumahnya dalam keadaan terkuncisetang serta menggunakan kunci pengaman tambahan lain berupa gembokyang dipasang apa piringan cakram depan, tetapi untuk kunci kontak sepedamotor tersebut masih tergantung tertinggal di lubang kunci jok sepeda motorsehingga terdakwa melakukan pencurian sepeda motor milik saksiHERIANSYAH Bin (Alm) H. Dengan demikian unsure ke5 telah terpenuhi ;Menimbang, bahwa berdasarkan faktafakta yang terungkap dalampersidangan, maka seluruh unsurunsur Pasal 362 KUHP telah terpenuhi, makaoleh karenanya Terdakwa telah terbukti secara sah dan meyakinkan bersalahmelakukan tindak pidana Pencurian, untuk itu. Menyatakan Terdakwa : DEDI KUMALA Bin SAHLAN telah terbukti secarasah dan menyakinkan bersalah melakukan tindak pidana PENCURIAN ;2. Menjatuhkan pidana terhadap Terdakwa DEDI KUMALA Bin SAHLAN,dengan pidana penjara selama : 1 (Satu) tahun dan 3 (tiga) bulan ;3."
  },
  {
    "no_perkara": "2000/Pid.B/2017/PN.Tng",
    "tanggal_putusan": "16 Nopember 2017",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/04ad52268ca0dad8d4bb28edf1b88c52.html",
    "title": "Putusan PN TANGERANG Nomor 2000/Pid.B/2017/PN.Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Pencurian Putus : 16-11-2017 — Upload : 20-12-2017 Putusan PN TANGERANG Nomor 2000/Pid.B/2017/PN.Tng Tanggal 16 Nopember 2017 — YUDI HARYADI Bin BAROZI. 36 — 1 Menyatakan Terdakwa YUDI HARYADI Bin BAROZI telah bersalahmelakukan tindak pidana Pencurian sebagaimana diatur dan diancampidana dalam Pasal 362 KUHP;2. Menjatuhkan pidana penjara terhadap Terdakwa YUDI HARYADI BinBAROZI tersebut dengan pidan penjara selama 1 (satu) tahun dikurangiselama terdakwa berada dalam tahanan dengan perintah Terdaka tetapditahan;3. Saksi DADANG SULASTOWO Bin CHAEROD4gJI Bahwa Terdakwa melakukan tindak pidana tersebut Pada hari Jumattanggal 11 agustus 2017 sekitar jam 12.20 wib pada saat sayasedang solat jumat, saya kehilangan laptop ditempat tugas saya diSMPN Cisoka Tangerang; Bahwa yang menjadi korban pencurian tersebut adalah saksi sendiri; Bahwa pada saat kejadian saksi sedang solat Jumat; Bahwa selain saksi masih ada 2 (dua) orang teman saksi yangmenjadi korban yaitu Anwar Syaddad dan Jarkasih yang jugakehilangan Laptopnya MADALI Bahwa Terdakwa melakukan tindak pidana tersebut Pada hari Jumattanggal 11 agustus 2017 sekitar jam 12.20 wib pada saat sayasedang solat jumat, saya kehilangan laptop ditempat tugas saya diSMPN Cisoka Tangerang; Bahwa yang menjadi korban pencurian tersebut adalah saksi sendiri; Bahwa pada saat kejadian saksi sedang solat Jumat; Bahwa selain saksi masih ada 2 (dua) orang teman saksi yangmenjadi korban yaitu Dadang dan Jarkasih yang juga kehilanganLaptopnya; Bahwa barang yang diambil oleh Terdakaw ABDLU FATAH (Alm) Bahwa Terdakwa melakukan tindak pidana tersebut Pada hari Jumattanggal 11 agustus 2017 sekitar jam 12.20 wib pada saat sayasedang solat jumat, saya kehilangan laptop ditempat tugas saya diSMPN Cisoka Tangerang; Bahwa yang menjadi korban pencurian tersebut adalah saksi sendiri; Bahwa pada saat kejadian saksi sedang solat Jumat; Bahwa selain saksi masih ada 2 (dua) orang teman saksi yangmenjadi korban yaitu Anwar Syaddad dan Dadang yang jugakehilangan Laptopnya; Bahwa barang yang Menyatakan Terdakwa Yudi Haryadi Bin Barozi telah bersalah melakukantindak pidana pencurian sebagaimana diatur dan diancam dalam Pasal 362KUHP;2. Menjatuhkan pidana kepada Terdakwa Yudi Haryadi Bin Barozi tersebutdengan pidana penjara selama 8 (delapan) bulan;3. Menetapkan masa penangkapan dan penahanan yang telah dijalani olehTerdakwa dikurangkan seluruhnya dari pidana yang dijatuhkan ;4. Memerintahkan agar Terdakwa tetap berada dalam tahanan ;5."
  },
  {
    "no_perkara": "2497/Pid.B/2018/PN Tng",
    "tanggal_putusan": "10 Januari 2019",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/2ec4a38929c18f44ab0e3893ecdd3020.html",
    "title": "Putusan PN TANGERANG Nomor 2497/Pid.B/2018/PN Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Register : 04-12-2018 — Putus : 10-01-2019 — Upload : 30-01-2019 Putusan PN TANGERANG Nomor 2497/Pid.B/2018/PN Tng Tanggal 10 Januari 2019 — Penuntut Umum: ROSI PAREME DEWI INDAH, SH Terdakwa: 1.AFRIYAZI Als AJI Bin EKO EDI PRIYANTO 2.WAHYU HIDAYATULLAH Bin SUANDI 36 — 3 MENGADILI: Menyatakan Terdakwa AFRIYAZI ALS AJI BIN EKO EDI PRIYANTO bersama dengan Terdakwa WAHYU HIDAYATULLAH BIN SUANDI bersalah melakukan tindak pidana Pencurian dengan kekerasan."
  },
  {
    "no_perkara": "2372/Pid.B/2018/PN Tng",
    "tanggal_putusan": "19 Desember 2018",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/1a2dbe4dff7e10c16022d7d40f3b8d54.html",
    "title": "Putusan PN TANGERANG Nomor 2372/Pid.B/2018/PN Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": false,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Register : 22-11-2018 — Putus : 19-12-2018 — Upload : 23-01-2019 Putusan PN TANGERANG Nomor 2372/Pid.B/2018/PN Tng Tanggal 19 Desember 2018 — Penuntut Umum: ANDRY SUDARMAJI, SH Terdakwa: ACHMAD MUDZAKIR BIN DADAN SULAEMAN 63 — 2"
  },
  {
    "no_perkara": "329/Pid.B/2019/PN Tng",
    "tanggal_putusan": "1 April 2019",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/1927e380e77376c28fcc56bec49431a4.html",
    "title": "Putusan PN TANGERANG Nomor 329/Pid.B/2019/PN Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Register : 19-02-2019 — Putus : 01-04-2019 — Upload : 06-09-2019 Putusan PN TANGERANG Nomor 329/Pid.B/2019/PN Tng Tanggal 1 April 2019 — Penuntut Umum: TRIADE Terdakwa: 1.ICON SIREGAR Alias ICON Bin SAMSIR ALAM SIREGAR 2.ADITYA Alias ADIT Bin KHAIDIR 3.DENI SELIAN Alias DEN Bin BUDIN 72 — 7 MENGADILI: Menyatakan bahwa Terdakwa I ICON SIREGAR Alias ICON Bin SAMSIR ALAM SIREGAR Terdakwa II ADITYA Alias ADIT Bin KHAIDIR Terdakwa III DENI SELIAN Alias DEN Bin BUDIN telahterbukti secara sah dan meyakinkan bersalah melakukan tindak pidana Pencurian dalam keadaan memberatkan; Menjatuhkan pidana terhadap Para Terdakwa tersebut diatas oleh karena itu dengan pidana penjara masing-masing selama 1(satu) tahun; Menetapkan lamanya Saksi TRIO SUWONDO, dibawah sumpah pada pokoknyamenerangkan sebagai berikut:Bahwa saksi menyerahkan mobil saksi Abdul Latif kepada terdakwauntuk disewakan selama dua hari untuk keperluan keluarga dan tidakmengetahui perihal aksi pencurian tersebut;Terhadap keterangan saksi Para Terdakwa membenarkan semuaketerangan Saksi.7. BI798WZBmerk Suzuki Ertiga warna abuabu metalik, lalu parkir didepan warnetbawah Uy over ciputat, kemudian lerdakwa 1 ICON SIREGAR,Terdakwa II ADITYA dan Terdakwa III DENI SELIAN masuk kedalammobil dan melihat Terdakwa 111 DENI membawa (satu) buah lasselempang hitam berisi kuncikunei, obeng dan sejenisnya, kKemudianberangkat dan Terdakwa ICON SIREGAR meminta saksi untukmengemudi dengan kecepatan 40 s/d 50 km/jam dengan tujuan agardapat melihat tempat tujuan unluk melakukan aksi pencurian tersebut BI798WZBmerk Suzuki Ertiga warna abuabu metalik, lalu parkir didepan warnetbawah Uy over ciputat, kemudian lerdakwa 1 ICON SIREGAR,Terdakwa Il ADITYA dan Terdakwa III DENI SELIAN masuk kedalammobil dan melihat Terdakwa 111 DENI membawa (satu) buah lasselempang hitam berisi kuncikunei, obeng dan sejenisnya, kKemudianberangkat dan Terdakwa ICON SIREGAR meminta saksi untukmengemudi dengan kecepatan 40 s/d 50 km/jam dengan tujuan agardapat melihat tempat tujuan unluk melakukan aksi pencurian tersebut Menyatakan bahwa Terdakwa ICON SIREGAR Alias ICON BinSAMSIR ALAM SIREGAR Terdakwa II ADITYA Alias ADIT Bin KHAIDIRTerdakwa III DENI SELIAN Alias DEN Bin BUDIN telahterbukti secara sahdan meyakinkan bersalah melakukan tindak pidana Pencurian dalamkeadaan memberatkan;2. Menjatuhkan pidana terhadap Para Terdakwa tersebut diatas olehkarena itu dengan pidana penjara masingmasing selama 1(satu) tahun;3."
  },
  {
    "no_perkara": "976/Pid.B/2022/PN Tng",
    "tanggal_putusan": "10 Agustus 2022",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed1ecc94bf613ebb5c313530353533.html",
    "title": "Putusan PN TANGERANG Nomor 976/Pid.B/2022/PN Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Register : 21-06-2022 — Putus : 10-08-2022 — Upload : 18-08-2022 Putusan PN TANGERANG Nomor 976/Pid.B/2022/PN Tng Tanggal 10 Agustus 2022 — Penuntut Umum: NGUNGUN ALIA SODIQ, SH Terdakwa: SUPRIYATNA Als CIMENG Bin DURASIP 37 — 9 MENGADILI Menyatakan Terdakwa SUPRIYATNA Alias CIMENG Bin DURASIP terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana Pencurian dalam keadaan memberatkan ."
  },
  {
    "no_perkara": "159/Pid.B/2022/PN Tng",
    "tanggal_putusan": "15 Maret 2022",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/zaeca5d173b3ce2282ba313630353534.html",
    "title": "Putusan PN TANGERANG Nomor 159/Pid.B/2022/PN Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Register : 02-02-2022 — Putus : 15-03-2022 — Upload : 17-03-2022 Putusan PN TANGERANG Nomor 159/Pid.B/2022/PN Tng Tanggal 15 Maret 2022 — Penuntut Umum: ENDAH KUSUMANINGTYAS, SH Terdakwa: RENDI SYAPUTRA als.RENDI Bin MULYADI 24 — 0 MENGADILI: Menyatakan Terdakwa RENDI SYAPUTRA Alias RENDI Bin MULYADI telah terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana pencurian dalam keadaan memberatkan, sebagaimana dakwaan alternatif pertama; Menjatuhkan pidana terhadap Terdakwa oleh karena itu dengan pidana penjara selama 1 (satu) tahun; Menetapkan masa penangkapan dan penahanan yang telah dijalani Terdakwa dikurangkan seluruhnya dari pidana yang dijatuhkan;"
  },
  {
    "no_perkara": "2013/Pid.B/2021/PN Tng",
    "tanggal_putusan": "9 Februari 2022",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/zaec8a47bfe91932a8d3313530323130.html",
    "title": "Putusan PN TANGERANG Nomor 2013/Pid.B/2021/PN Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Register : 16-12-2021 — Putus : 09-02-2022 — Upload : 10-02-2022 Putusan PN TANGERANG Nomor 2013/Pid.B/2021/PN Tng Tanggal 9 Februari 2022 — Penuntut Umum: DINA NATALIA, SH Terdakwa: NICKY KRISTIAWAN ALS SENGSENG AD. PING CUAN 63 — 12 MENGADILI: Menyatakan Terdakwa NICKY KRISTIAWAN Als SENGSENG A.d PING CUAN bersalah melakukan tindak pidana Pencurian Dalam keadaan memberatkan."
  },
  {
    "no_perkara": "960/Pid.B/2022/PN Tng",
    "tanggal_putusan": "2 Agustus 2022",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed1240ef069f92a5a6313535363031.html",
    "title": "Putusan PN TANGERANG Nomor 960/Pid.B/2022/PN Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Register : 16-06-2022 — Putus : 02-08-2022 — Upload : 02-08-2022 Putusan PN TANGERANG Nomor 960/Pid.B/2022/PN Tng Tanggal 2 Agustus 2022 — Penuntut Umum: DAVID ANDI, SH Terdakwa: 1.MUHAMMAD NURDIN Als KIKI Bin DEDI KUSNAEDI 2.BAGAS ABDHY NUGRAHA Als DINYO DONO Bin SAHRON SUSILO 3.EDO RAFAEL FAUZAN Bin RIKI FAUZAN 39 — 9 Edo Rafael Fauzan Bin Riki Fauzan, tersebut diatas, terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana Pencurian dengan kekerasan; Menjatuhkan pidana kepada Para Terdakwa oleh karena itu dengan pidana penjara masing-masing selama 3 (tiga) Tahun; Menetapkan masa penangkapan dan penahanan yang telah dijalani Para Terdakwa dikurangkan seluruhnya dari pidana yang dijatuhkan; Menetapkan Para Terdakwa tetap ditahan; Menetapkan barang"
  },
  {
    "no_perkara": "1427/Pid.B/2022/PN Tng",
    "tanggal_putusan": "16 Nopember 2022",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed748157d4a90492eb313634333539.html",
    "title": "Putusan PN TANGERANG Nomor 1427/Pid.B/2022/PN Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Register : 31-08-2022 — Putus : 16-11-2022 — Upload : 05-12-2022 Putusan PN TANGERANG Nomor 1427/Pid.B/2022/PN Tng Tanggal 16 Nopember 2022 — Penuntut Umum: EKO PURWANTO, SH Terdakwa: HASAN FUAD Bin DAYAT 41 — 1 MENGADILI: Menyatakan Terdakwa Hasan Fuad Bin Dayat tersebut diatas, terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana Pencurian dalam keadaan memberatkan; Menjatuhkan pidana kepada Terdakwa oleh karena itu dengan pidana penjara selama 2 (dua) tahun; Menetapkan masa penangkapan dan penahanan yang telah dijalani Terdakwa dikurangkan seluruhnya dari pidana yang dijatuhkan; Menetapkan"
  },
  {
    "no_perkara": "1471/Pid.B/2022/PN Tng",
    "tanggal_putusan": "19 Oktober 2022",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/zaed5053a547779a910e313534363130.html",
    "title": "Putusan PN TANGERANG Nomor 1471/Pid.B/2022/PN Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": false,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Register : 06-09-2022 — Putus : 19-10-2022 — Upload : 20-10-2022 Putusan PN TANGERANG Nomor 1471/Pid.B/2022/PN Tng Tanggal 19 Oktober 2022 — Penuntut Umum: MEFFY OLIVIA, SH Terdakwa: 1.RIDWAN EFFENDI alias WAWAN bin alm H. SANIM 2.IWAN SETIAWAN alias TICUL bin ROHMAN 30 — 3"
  },
  {
    "no_perkara": "1176/Pid.B/2018/PN Tng",
    "tanggal_putusan": "19 Juli 2018",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/0da4d46117db2e9b1568b5dfe9f38697.html",
    "title": "Putusan PN TANGERANG Nomor 1176/Pid.B/2018/PN Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Register : 04-06-2018 — Putus : 19-07-2018 — Upload : 28-11-2018 Putusan PN TANGERANG Nomor 1176/Pid.B/2018/PN Tng Tanggal 19 Juli 2018 — Penuntut Umum: HERU APRIANTO, SH Terdakwa: ISEP ISKANDAR Bin ISHAK 39 — 3 M E N G A D I L I : Menyatakan terdakwa ISEP ISKANDAR Bin ISHAK terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana Percobaan Pencurian dalam keadaan memberatkan ; Menjatuhkan pidana terhadap terdakwa dengan pidana penjara selama 10 (sepuluh) bulan; Menetapkan masa penangkapan dan penahanan yang telah dijalani Terdakwa dikurangkan seluruhnya dari pidana yang dijatuhkan; Menetapkan Terdakwa Tangerang.Bahwa yang menjadi pelaku pencurian tersebut adalah ISEP ISKANDAR, danyang menjadi korbannya adalah Saksi sendiri.Bahwa 1 (Satu) Unit Sepeda Motor merk Honda Vario tahun 2013, warnaputih silver, Nopol : B 6586 CWG, Noka : MH1JFB113DK756965, Nosin :JFB1E1714364, an Sdr DONAASMARA.Bahwa terdakwa melakukan Tindak pidana Pencurian tersebut dengan caramengambil motor saksi yang sedang terparkir di depan warung makan yangberalamatkan di Kp. Sumur Bandung Ds.Sumur bandung Kec. Saksi FERDI PANGRINGGO bin SUDARLI. didepan persidangan dibawah sumpah Pada pokoknyamenerangkan sebagai berikut:Bahwa kejadian Pencurian tersebut terjadi pada hari Jumat tanggal 30 Maret 2018 sekiranya jam 17.00Wib di warung makan yang beralamatkan di Kp. Sumur Bandung Ds. Sumur bandung Kec. Jayanti Kab.Tangerang.Bahwa yang menjadi pelaku pencurian tersebut adalah ISEP ISKANDAR, dan yang menjadi korbannyaadalah Saksi sendiri.Bahwa 1 (Satu) Unit Sepeda Motor merk Honda Vario tahun 2013, warna putih silver, Nopol : B 6586CWG, Noka : MH1JFB113DK756965, Nosin : JFB1E1714364, an Sdr DONA ASMARA.Bahwa terdakwa melakukan Tindak pidana Pencurian tersebut dengan cara mengambil motor saksiyang sedang terparkir di depan warung makan yang beralamatkan di Kp. Sumur Bandung Ds. Sumurbandung Kec. Bahwa total kerugian yang dialami oleh saksi akibat dari pencurian tersebut adalah sebesar Rp.10.500.000 (sepuluh juta lima rafus ribu rupiah).Atas keterangan saksi tersebut, terdakwa tidak keberatan dan membenarkaimya2. Bahwa sebelumnya saksi tidak tahu nama pelaku yang telah melakukan pencurian sepeda motor tersebut,akan tertapi setelah saksi menanyakan kepada pelaku, bahwa pelaku pencurian sepeda motor tersebutbernama : ISEP ISKANDAR, umur : 38 Th, alamat : Ds Bungku Rt 03/05 Ds Bungku, Kec, WargaSekampung, Kab, Lampung Timur, dan yang menjadi korban adalah sdr ferdi pangringggo UMUR : 26Th, alamat: Perum Bukit Gading balaraja Blok A4 No 10, Ds Ciapus, Kec, Balaraja, Kab Tangerang."
  },
  {
    "no_perkara": "586/Pid.B/2019/PN Tng",
    "tanggal_putusan": "16 Mei 2019",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/08310eb6ee84116780cee64575f7054e.html",
    "title": "Putusan PN TANGERANG Nomor 586/Pid.B/2019/PN Tng",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Register : 21-03-2019 — Putus : 16-05-2019 — Upload : 22-05-2019 Putusan PN TANGERANG Nomor 586/Pid.B/2019/PN Tng Tanggal 16 Mei 2019 — Penuntut Umum: ANDRE SAUT SIMBOLON, SH Terdakwa: 1.SANWANI Alisa OLOT Bin JUHANI alm 2.JUMAN Bin ACIM 3.AYON KUNIAWAN Als AYON Bin SARIAN 59 — 53 AYON KURNIAWAN Als AYON Bin SARIAN, telah terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana : \"Pencurian dalam keadaan memberatkan yang dilakukan secara bersama-sama; Menjatuhkan pidana kepada Terdakwa I. SANWANI Als OLOT Bin JUHANI, Terdakwa II. JUMAN Bin ACIN dan Terdakwa III. Saksi: HENRY JAYA KNEE,Bahwa benar Saksi kenal dengan Para Terdakwa karena Para Terdakwakaryawan saksi;;Bahwa Telah terjadi Tindak Pidana Pencurian pada hari Minggu tanggal30 Desember 2018 sekira pukul 02.00 Wib yang dilakukan oleh ParaTerdakwa pada Perternakan CV. Berlian Jaya Farm milik Saksi di Kp.Cirarab Kec. Legok Kab. tidak mengetahui kapan dan bagaimana cara ParaTerdakwa melakukan pencurian tersebut kejadian tersebut tanpasepengetahuan dan seizin dari Saksi;Bahwa akibat dari pencurian yang dilakukan para Terdakwa sehinggaSaksi mengalami kerugian sebesar Rp.7.800.000, (tujuh juta delapanratus ribu rupiah);Bahwa atas peristiwa tersebut untuk selanjutnya, Saksi melaporkan haltersebut kepihak yang berwajib untuk diproses lebih lanjut.Menimbang, bahwa atas keterangan saksi tersebut para Terdakwa telahmembenarkannya tersebut dari Saksi HENRY JAYAKWEE yang menemukan selisih ketika dilakukan audit;Bahwa Saksi tidak mengetahui kapan dan bagaimana cara ParaTerdakwa melakukan pencurian pakan ternak tersebut kejadian tersebuttanpa sepengetahuan dan seizin dari Saksi HENRY JAYA KWEEsehingga CV. Berlian Jaya Farm;Bahwa yang diruri oleh para Terdakwa yaitu 200 ekor ayam dan 4 karungpakan ayam;Bahwa ata kejadian pencurian tersebut mengalami kerugian sebesarRp.7.800.000, (tujuh juta delapan ratus ribu rupiah)Halctman 5 dari 16, PutusanNomor 586/Pid.B/2019/PN. Tangerang;Bahwa benar awalnya Saksi mengetahui hal tersebut dari Saksi HENRYJAYA KWEE yang menemukan selisih ketika dilakukan audit;Bahwa Saks tidak mengetahui kapan dan bagaimana cara Para Terdakwamelakukan pencurian tersebut kejadian tersebut tanpa sepengetahuandan seizin dari Saksi HENRY JAYA KNEE sehingga CV."
  },
  {
    "no_perkara": "2203/PID.B/2014/PN. TNG",
    "tanggal_putusan": "6 Januari 2015",
    "pengadilan": "PN Tangerang",
    "jenis_perkara": "Pidana Umum - Pencurian",
    "url": "https://putusan3.mahkamahagung.go.id/direktori/putusan/84cb49872067c701cfdf1ec9ba18e2d7.html",
    "title": "Putusan PN TANGERANG Nomor 2203/PID.B/2014/PN. TNG",
    "is_pn_tangerang": true,
    "is_pidana_umum": true,
    "is_pencurian": true,
    "card_text": "Pengadilan PN TANGERANG Pidana Umum Pencurian Putus : 06-01-2015 — Upload : 15-06-2015 Putusan PN TANGERANG Nomor 2203/PID.B/2014/PN. TNG Tanggal 6 Januari 2015 — ADI PRIHANTORO als GALANG Bin BAMBANG PRAWOTO 35 — 7 tuntutan pidana yang diajukan olehPenuntut Umum yang pada pokoknya sebagai berikut:1.Menyatakan terdakwa ADI PRIHANTORO Als GALANG Bin BAMBANGPRAWOTO terbukti secara sah dan meyakinkan menurut hukum bersalahtelah melakukan tindak pidana mengambil barang sesuatu yangseluruhnya atau sebagian kepunyaan orang lain dengan maksud untukdimiliki secara melawan hukum, yang didahului, disertai atau diikutidengan kekerasan atau ancaman kekerasan, terhadap orang denganmaksud untuk mempersiapkan atau mempermudah pencurian Bambu ApusPamulang Tangerang Selatan atau setidaktidaknya pada suatu tempattempattertentu yang masih termasuk dalam daerah Hukum Pengadilan NegeriTangerang yang berwenang memeriksa dan mengadili, \"Mengambil barangsesuatu yang seluruhnya atau sebagian kepunyaan orang lain denganmaksud untuk dimiliki secara melawan hukum, yang didahului, disertai atau diikutidengan kekerasan atau ancaman kekerasan terhadap orang, dengan maksuduntuk mempersiapkan atau mempermudah pencurian, dalam hal tertangkaptangan Pamulang Tangerang Selatan terdakwa telahmelakukan pencurian dengan kekerasan pada anak saksi yangbernama KAMILA FAIRUZ .Bahwa barang yang diambil oleh terdakwa adalah 1 (satu) buah handphoen merek NOKIA X.2 millik anak saksi KAMILA FAIRUZ.bahwa secara pasti saksi tidak mengetahui bagaimana caranyaterdakwa melakukan pencurian dengan kekerasan tetapi menurutketerangan anak saksi KAMILA FAIRUZ bahwa pelaku melakukanpencurian dengan kekerasan dengancara pertama berkenalandengananak saksi di face book Bahwa dengan kejadian pencurian tersebut kerugiannyasebesar Rp. 750.000,00 (tujuh ratus lima puluh ribu rupiah bahwa saksi di visum dan saksi mengalami luka lecet padapipi kanan, pada daerah leher pada dahi kiri. Terhadap keterangan saksi, Terdakwa memberikan pendapat tidakkeberatan terhadap keterangan saksi tersebut;Menimbang, bahwa dipersidangan didengar pula keterangan saksi yangtidak hadir yaitu :3. Menyatakan Terdakwa ADI PRIHANTORO als GALANG Bin BAMBANGPRAWOTO tersebut diatas terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana Pencurian Dengan Kekerasan ;2. Menjatuhkan pidana kepada Terdakwa oleh karena itu dengan pidanapenjara selama 1 (satu) tahun dan 6 (enam) bulan;3."
  }
]
"""

In [10]:
from pathlib import Path
import pandas as pd
import json

BASE_DIR = Path("..").resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

data_page1 = json.loads(page1_json)

df_page1 = pd.DataFrame(data_page1)

print("Jumlah data mentah page 1:", len(df_page1))

df_page1[[
    "no_perkara",
    "tanggal_putusan",
    "pengadilan",
    "jenis_perkara",
    "url",
    "is_pn_tangerang",
    "is_pidana_umum",
    "is_pencurian"
]]

Jumlah data mentah page 1: 20


,no_perkara,tanggal_putusan,pengadilan,jenis_perkara,url,is_pn_tangerang,is_pidana_umum,is_pencurian
0,1022/Pid.B/2010/PN.TNG,14 Juli 2010,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,True,True,True
1,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,True,True,True
2,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,True,True,True
3,1073/Pid.B/2019/PN Tng,10 Juli 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,True,True,True
4,678/ PID.B/ 2011/ PN TNG,8 Mei 2012,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,True,True,True
5,1527/Pid.B/2014/PN.TNG,16 September 2014,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,True,True,True
6,834 / Pid.B / 2017 / PN.TNG.,15 Juni 2017,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,True,True,True
7,2000/Pid.B/2017/PN.Tng,16 Nopember 2017,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,True,True,True
8,2497/Pid.B/2018/PN Tng,10 Januari 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,True,True,True
9,2372/Pid.B/2018/PN Tng,19 Desember 2018,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,True,True,False


In [11]:
df_valid_page1 = df_page1[
    (df_page1["is_pn_tangerang"] == True) &
    (df_page1["is_pidana_umum"] == True) &
    (df_page1["is_pencurian"] == True)
].copy()

df_valid_page1 = df_valid_page1.drop_duplicates(subset=["url"]).reset_index(drop=True)

print("Jumlah data valid page 1:", len(df_valid_page1))

df_valid_page1[[
    "no_perkara",
    "tanggal_putusan",
    "pengadilan",
    "jenis_perkara",
    "url"
]]

Jumlah data valid page 1: 18


,no_perkara,tanggal_putusan,pengadilan,jenis_perkara,url
0,1022/Pid.B/2010/PN.TNG,14 Juli 2010,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...
1,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...
2,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...
3,1073/Pid.B/2019/PN Tng,10 Juli 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...
4,678/ PID.B/ 2011/ PN TNG,8 Mei 2012,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...
5,1527/Pid.B/2014/PN.TNG,16 September 2014,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...
6,834 / Pid.B / 2017 / PN.TNG.,15 Juni 2017,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...
7,2000/Pid.B/2017/PN.Tng,16 Nopember 2017,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...
8,2497/Pid.B/2018/PN Tng,10 Januari 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...
9,329/Pid.B/2019/PN Tng,1 April 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...


In [12]:
df_inventory_page1 = df_valid_page1[[
    "no_perkara",
    "tanggal_putusan",
    "pengadilan",
    "jenis_perkara",
    "url"
]].copy()

df_inventory_page1 = df_inventory_page1.rename(columns={
    "url": "sumber_url"
})

df_inventory_page1.insert(0, "case_id", [f"case_{i:03d}" for i in range(1, len(df_inventory_page1) + 1)])

df_inventory_page1["raw_file"] = df_inventory_page1["case_id"] + ".txt"
df_inventory_page1["status_download"] = "belum"
df_inventory_page1["jumlah_kata"] = "0"

page1_inventory_path = PROCESSED_DIR / "case_inventory_page1.csv"

df_inventory_page1.to_csv(page1_inventory_path, index=False)

print("Inventory Page 1 berhasil dibuat:")
print(page1_inventory_path)

df_inventory_page1

Inventory Page 1 berhasil dibuat:
/home/zack/Penalaran-Komputer-subcpmk-3/data/processed/case_inventory_page1.csv


,case_id,no_perkara,tanggal_putusan,pengadilan,jenis_perkara,sumber_url,raw_file,status_download,jumlah_kata
0,case_001,1022/Pid.B/2010/PN.TNG,14 Juli 2010,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_001.txt,belum,0
1,case_002,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_002.txt,belum,0
2,case_003,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_003.txt,belum,0
3,case_004,1073/Pid.B/2019/PN Tng,10 Juli 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_004.txt,belum,0
4,case_005,678/ PID.B/ 2011/ PN TNG,8 Mei 2012,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_005.txt,belum,0
5,case_006,1527/Pid.B/2014/PN.TNG,16 September 2014,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_006.txt,belum,0
6,case_007,834 / Pid.B / 2017 / PN.TNG.,15 Juni 2017,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_007.txt,belum,0
7,case_008,2000/Pid.B/2017/PN.Tng,16 Nopember 2017,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_008.txt,belum,0
8,case_009,2497/Pid.B/2018/PN Tng,10 Januari 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_009.txt,belum,0
9,case_010,329/Pid.B/2019/PN Tng,1 April 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_010.txt,belum,0


In [13]:
from pathlib import Path

BASE_DIR = Path("..").resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"

page1_path = PROCESSED_DIR / "case_inventory_page1.csv"

df_inventory_page1.to_csv(page1_path, index=False)

print("Berhasil disimpan:")
print(page1_path)

Berhasil disimpan:
/home/zack/Penalaran-Komputer-subcpmk-3/data/processed/case_inventory_page1.csv


In [14]:
from pathlib import Path
import pandas as pd
import json

BASE_DIR = Path("..").resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"

page1_path = PROCESSED_DIR / "case_inventory_page1.csv"
script_path = BASE_DIR / "browser_extract_page1_batch1.js"

df = pd.read_csv(page1_path, dtype=str).fillna("")

# Ambil 5 data pertama dulu untuk testing
batch_df = df.iloc[0:5].copy()

records = batch_df[[
    "case_id",
    "no_perkara",
    "tanggal_putusan",
    "sumber_url",
    "raw_file"
]].rename(columns={"sumber_url": "url"}).to_dict(orient="records")

js_code = f"""
(async () => {{
  const rows = {json.dumps(records, ensure_ascii=False, indent=2)};

  const sleep = (ms) => new Promise(resolve => setTimeout(resolve, ms));

  const cleanText = (text) => {{
    return text
      .replace(/\\r/g, "\\n")
      .replace(/\\t/g, " ")
      .replace(/\\n{{3,}}/g, "\\n\\n")
      .replace(/[ ]{{2,}}/g, " ")
      .trim();
  }};

  const isSecurityPage = (text) => {{
    const lower = text.toLowerCase();
    return (
      lower.includes("cloudflare") ||
      lower.includes("enable javascript and cookies") ||
      lower.includes("melakukan verifikasi keamanan") ||
      lower.includes("ray id") ||
      lower.includes("bot jahat")
    );
  }};

  const results = [];

  console.log("Mulai mengambil detail HTML batch 1:", rows.length, "kasus");

  for (let i = 0; i < rows.length; i++) {{
    const row = rows[i];

    console.log(`[${{i + 1}}/${{rows.length}}] Memproses ${{row.case_id}}`);

    try {{
      const response = await fetch(row.url, {{
        credentials: "include",
        cache: "no-store"
      }});

      const html = await response.text();
      const doc = new DOMParser().parseFromString(html, "text/html");

      doc.querySelectorAll("script, style, nav, footer, header").forEach(el => el.remove());

      let text = "";

      if (doc.body) {{
        text = doc.body.innerText || doc.body.textContent || "";
      }}

      text = cleanText(text);

      const wordCount = text.length > 0 ? text.split(/\\s+/).filter(Boolean).length : 0;

      let status = "berhasil_html";

      if (response.status !== 200) {{
        status = "gagal_http_" + response.status;
      }} else if (isSecurityPage(text)) {{
        status = "gagal_cloudflare";
      }} else if (wordCount < 50) {{
        status = "teks_pendek";
      }}

      results.push({{
        case_id: row.case_id,
        no_perkara: row.no_perkara,
        tanggal_putusan: row.tanggal_putusan,
        raw_file: row.raw_file,
        url: row.url,
        status: status,
        http_status: response.status,
        jumlah_kata: wordCount,
        text: text
      }});

      console.log(row.case_id, status, wordCount, "kata");

    }} catch (err) {{
      results.push({{
        case_id: row.case_id,
        no_perkara: row.no_perkara,
        tanggal_putusan: row.tanggal_putusan,
        raw_file: row.raw_file,
        url: row.url,
        status: "gagal_error",
        error: String(err),
        jumlah_kata: 0,
        text: ""
      }});

      console.error("Gagal:", row.case_id, err);
    }}

    await sleep(4000);
  }}

  const blob = new Blob([JSON.stringify(results, null, 2)], {{
    type: "application/json"
  }});

  const a = document.createElement("a");
  a.href = URL.createObjectURL(blob);
  a.download = "ma_putusan_page1_batch1.json";
  document.body.appendChild(a);
  a.click();
  document.body.removeChild(a);

  console.log("Selesai. File ma_putusan_page1_batch1.json sudah dibuat/download.");
}})();
"""

script_path.write_text(js_code, encoding="utf-8")

print("Script batch 1 berhasil dibuat:")
print(script_path)

Script batch 1 berhasil dibuat:
/home/zack/Penalaran-Komputer-subcpmk-3/browser_extract_page1_batch1.js


In [15]:
from pathlib import Path
import pandas as pd
import json

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

json_path = Path.home() / "Downloads" / "ma_putusan_page1_cards.json"

if not json_path.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {json_path}")

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

df_page1 = pd.DataFrame(data)

print("Jumlah data mentah:", len(df_page1))

df_valid = df_page1[
    (df_page1["is_pn_tangerang"] == True) &
    (df_page1["is_pidana_umum"] == True) &
    (df_page1["is_pencurian"] == True)
].copy()

df_valid = df_valid.drop_duplicates(subset=["url"]).reset_index(drop=True)

print("Jumlah data valid:", len(df_valid))

inventory_rows = []

for i, row in df_valid.iterrows():
    case_id = f"case_{i+1:03d}"
    raw_file = f"{case_id}.txt"
    raw_path = RAW_DIR / raw_file

    text = row.get("text", "")
    raw_path.write_text(text, encoding="utf-8")

    inventory_rows.append({
        "case_id": case_id,
        "no_perkara": row.get("no_perkara", ""),
        "tanggal_putusan": row.get("tanggal_putusan", ""),
        "pengadilan": "PN Tangerang",
        "jenis_perkara": "Pidana Umum - Pencurian",
        "sumber_url": row.get("url", ""),
        "raw_file": raw_file,
        "status_download": "berhasil_list_html",
        "jumlah_kata": row.get("jumlah_kata", 0)
    })

df_inventory_page1 = pd.DataFrame(inventory_rows)

page1_inventory_path = PROCESSED_DIR / "case_inventory_page1.csv"
df_inventory_page1.to_csv(page1_inventory_path, index=False)

print("Inventory Page 1 berhasil dibuat:")
print(page1_inventory_path)

print("File raw txt berhasil dibuat di:")
print(RAW_DIR)

df_inventory_page1

Jumlah data mentah: 20
Jumlah data valid: 18
Inventory Page 1 berhasil dibuat:
/home/zack/Penalaran-Komputer-subcpmk-3/data/processed/case_inventory_page1.csv
File raw txt berhasil dibuat di:
/home/zack/Penalaran-Komputer-subcpmk-3/data/raw


,case_id,no_perkara,tanggal_putusan,pengadilan,jenis_perkara,sumber_url,raw_file,status_download,jumlah_kata
0,case_001,1022/Pid.B/2010/PN.TNG,14 Juli 2010,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_001.txt,berhasil_list_html,80
1,case_002,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_002.txt,berhasil_list_html,322
2,case_003,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_003.txt,berhasil_list_html,123
3,case_004,1073/Pid.B/2019/PN Tng,10 Juli 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_004.txt,berhasil_list_html,213
4,case_005,678/ PID.B/ 2011/ PN TNG,8 Mei 2012,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_005.txt,berhasil_list_html,342
5,case_006,1527/Pid.B/2014/PN.TNG,16 September 2014,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_006.txt,berhasil_list_html,276
6,case_007,834 / Pid.B / 2017 / PN.TNG.,15 Juni 2017,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_007.txt,berhasil_list_html,263
7,case_008,2000/Pid.B/2017/PN.Tng,16 Nopember 2017,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_008.txt,berhasil_list_html,349
8,case_009,2497/Pid.B/2018/PN Tng,10 Januari 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_009.txt,berhasil_list_html,73
9,case_010,329/Pid.B/2019/PN Tng,1 April 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_010.txt,berhasil_list_html,335


In [1]:
from pathlib import Path
import pandas as pd
import json

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

page1_inventory_path = PROCESSED_DIR / "case_inventory_page1.csv"
page2_inventory_path = PROCESSED_DIR / "case_inventory_page2.csv"
current_inventory_path = PROCESSED_DIR / "case_inventory_current.csv"

json_path = Path.home() / "Downloads" / "ma_putusan_page2_cards.json"

if not json_path.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {json_path}")

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

df_page2 = pd.DataFrame(data)

print("Jumlah data mentah page 2:", len(df_page2))

df_valid_page2 = df_page2[
    (df_page2["is_pn_tangerang"] == True) &
    (df_page2["is_pidana_umum"] == True) &
    (df_page2["is_pencurian"] == True)
].copy()

df_valid_page2 = df_valid_page2.drop_duplicates(subset=["url"]).reset_index(drop=True)

print("Jumlah data valid page 2:", len(df_valid_page2))

# Ambil data page 1 untuk menghindari duplikat dan melanjutkan nomor case_id
if page1_inventory_path.exists():
    df_page1_inventory = pd.read_csv(page1_inventory_path, dtype=str).fillna("")
else:
    df_page1_inventory = pd.DataFrame()

existing_urls = set()

if not df_page1_inventory.empty:
    existing_urls = set(df_page1_inventory["sumber_url"].tolist())

df_valid_page2 = df_valid_page2[~df_valid_page2["url"].isin(existing_urls)].copy()
df_valid_page2 = df_valid_page2.reset_index(drop=True)

print("Jumlah data valid page 2 setelah hapus duplikat:", len(df_valid_page2))

offset = len(df_page1_inventory)

inventory_rows = []

for i, row in df_valid_page2.iterrows():
    case_number = offset + i + 1
    case_id = f"case_{case_number:03d}"
    raw_file = f"{case_id}.txt"
    raw_path = RAW_DIR / raw_file

    text = row.get("text", "")
    raw_path.write_text(text, encoding="utf-8")

    inventory_rows.append({
        "case_id": case_id,
        "no_perkara": row.get("no_perkara", ""),
        "tanggal_putusan": row.get("tanggal_putusan", ""),
        "pengadilan": "PN Tangerang",
        "jenis_perkara": "Pidana Umum - Pencurian",
        "sumber_url": row.get("url", ""),
        "raw_file": raw_file,
        "status_download": "berhasil_list_html",
        "jumlah_kata": row.get("jumlah_kata", 0)
    })

df_inventory_page2 = pd.DataFrame(inventory_rows)

df_inventory_page2.to_csv(page2_inventory_path, index=False)

print("Inventory Page 2 berhasil dibuat:")
print(page2_inventory_path)

# Gabungkan Page 1 + Page 2
frames = []

if page1_inventory_path.exists():
    frames.append(pd.read_csv(page1_inventory_path, dtype=str).fillna(""))

frames.append(df_inventory_page2)

df_current = pd.concat(frames, ignore_index=True)
df_current = df_current.drop_duplicates(subset=["sumber_url"]).reset_index(drop=True)

df_current.to_csv(current_inventory_path, index=False)

print("Inventory gabungan sementara berhasil dibuat:")
print(current_inventory_path)

print("Total data sementara:", len(df_current))

df_current

Jumlah data mentah page 2: 20
Jumlah data valid page 2: 20
Jumlah data valid page 2 setelah hapus duplikat: 20
Inventory Page 2 berhasil dibuat:
/home/zack/Penalaran-Komputer-subcpmk-3/data/processed/case_inventory_page2.csv
Inventory gabungan sementara berhasil dibuat:
/home/zack/Penalaran-Komputer-subcpmk-3/data/processed/case_inventory_current.csv
Total data sementara: 38


,case_id,no_perkara,tanggal_putusan,pengadilan,jenis_perkara,sumber_url,raw_file,status_download,jumlah_kata
0,case_001,1022/Pid.B/2010/PN.TNG,14 Juli 2010,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_001.txt,berhasil_list_html,80
1,case_002,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_002.txt,berhasil_list_html,322
2,case_003,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_003.txt,berhasil_list_html,123
3,case_004,1073/Pid.B/2019/PN Tng,10 Juli 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_004.txt,berhasil_list_html,213
4,case_005,678/ PID.B/ 2011/ PN TNG,8 Mei 2012,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_005.txt,berhasil_list_html,342
5,case_006,1527/Pid.B/2014/PN.TNG,16 September 2014,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_006.txt,berhasil_list_html,276
6,case_007,834 / Pid.B / 2017 / PN.TNG.,15 Juni 2017,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_007.txt,berhasil_list_html,263
7,case_008,2000/Pid.B/2017/PN.Tng,16 Nopember 2017,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_008.txt,berhasil_list_html,349
8,case_009,2497/Pid.B/2018/PN Tng,10 Januari 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_009.txt,berhasil_list_html,73
9,case_010,329/Pid.B/2019/PN Tng,1 April 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_010.txt,berhasil_list_html,335


In [2]:
from pathlib import Path
import pandas as pd
import json

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

current_inventory_path = PROCESSED_DIR / "case_inventory_current.csv"
page3_inventory_path = PROCESSED_DIR / "case_inventory_page3.csv"
final_inventory_path = PROCESSED_DIR / "case_inventory.csv"

json_path = Path.home() / "Downloads" / "ma_putusan_page3_cards.json"

if not json_path.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {json_path}")

df_current = pd.read_csv(current_inventory_path, dtype=str).fillna("")

print("Total data sebelum Page 3:", len(df_current))

needed = 40 - len(df_current)

print("Sisa data yang dibutuhkan:", needed)

if needed <= 0:
    print("Data sudah 40 atau lebih. Tidak perlu tambah Page 3.")
else:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    df_page3 = pd.DataFrame(data)

    print("Jumlah data mentah Page 3:", len(df_page3))

    df_valid_page3 = df_page3[
        (df_page3["is_pn_tangerang"] == True) &
        (df_page3["is_pidana_umum"] == True) &
        (df_page3["is_pencurian"] == True)
    ].copy()

    df_valid_page3 = df_valid_page3.drop_duplicates(subset=["url"]).reset_index(drop=True)

    existing_urls = set(df_current["sumber_url"].tolist())

    df_valid_page3 = df_valid_page3[~df_valid_page3["url"].isin(existing_urls)].copy()
    df_valid_page3 = df_valid_page3.reset_index(drop=True)

    print("Jumlah data valid Page 3 setelah hapus duplikat:", len(df_valid_page3))

    # Ambil hanya sejumlah yang dibutuhkan
    df_valid_page3 = df_valid_page3.head(needed).copy()

    print("Jumlah data Page 3 yang akan ditambahkan:", len(df_valid_page3))

    offset = len(df_current)

    inventory_rows = []

    for i, row in df_valid_page3.iterrows():
        case_number = offset + i + 1
        case_id = f"case_{case_number:03d}"
        raw_file = f"{case_id}.txt"
        raw_path = RAW_DIR / raw_file

        text = row.get("text", "")
        raw_path.write_text(text, encoding="utf-8")

        inventory_rows.append({
            "case_id": case_id,
            "no_perkara": row.get("no_perkara", ""),
            "tanggal_putusan": row.get("tanggal_putusan", ""),
            "pengadilan": "PN Tangerang",
            "jenis_perkara": "Pidana Umum - Pencurian",
            "sumber_url": row.get("url", ""),
            "raw_file": raw_file,
            "status_download": "berhasil_list_html",
            "jumlah_kata": row.get("jumlah_kata", 0)
        })

    df_inventory_page3 = pd.DataFrame(inventory_rows)
    df_inventory_page3.to_csv(page3_inventory_path, index=False)

    print("Inventory Page 3 berhasil dibuat:")
    print(page3_inventory_path)

    df_final = pd.concat([df_current, df_inventory_page3], ignore_index=True)
    df_final = df_final.drop_duplicates(subset=["sumber_url"]).reset_index(drop=True)

    # Pastikan maksimal 40 data
    df_final = df_final.head(40).copy()

    df_final.to_csv(current_inventory_path, index=False)
    df_final.to_csv(final_inventory_path, index=False)

    print("Inventory final berhasil dibuat:")
    print(final_inventory_path)

    print("Total data final:", len(df_final))

    df_final

Total data sebelum Page 3: 38
Sisa data yang dibutuhkan: 2
Jumlah data mentah Page 3: 20
Jumlah data valid Page 3 setelah hapus duplikat: 18
Jumlah data Page 3 yang akan ditambahkan: 2
Inventory Page 3 berhasil dibuat:
/home/zack/Penalaran-Komputer-subcpmk-3/data/processed/case_inventory_page3.csv
Inventory final berhasil dibuat:
/home/zack/Penalaran-Komputer-subcpmk-3/data/processed/case_inventory.csv
Total data final: 40


In [3]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

final_inventory_path = PROCESSED_DIR / "case_inventory.csv"

df_final = pd.read_csv(final_inventory_path, dtype=str).fillna("")

print("Total data:", len(df_final))
print("Duplikat URL:", df_final["sumber_url"].duplicated().sum())
print("Jumlah raw file yang ada:", sum((RAW_DIR / f).exists() for f in df_final["raw_file"]))

df_final[[
    "case_id",
    "no_perkara",
    "tanggal_putusan",
    "pengadilan",
    "jenis_perkara",
    "sumber_url",
    "raw_file",
    "status_download",
    "jumlah_kata"
]]

Total data: 40
Duplikat URL: 0
Jumlah raw file yang ada: 40


,case_id,no_perkara,tanggal_putusan,pengadilan,jenis_perkara,sumber_url,raw_file,status_download,jumlah_kata
0,case_001,1022/Pid.B/2010/PN.TNG,14 Juli 2010,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_001.txt,berhasil_list_html,80
1,case_002,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_002.txt,berhasil_list_html,322
2,case_003,1885 /Pid.B/2011/PN.TNG,15 Desember 2011,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_003.txt,berhasil_list_html,123
3,case_004,1073/Pid.B/2019/PN Tng,10 Juli 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_004.txt,berhasil_list_html,213
4,case_005,678/ PID.B/ 2011/ PN TNG,8 Mei 2012,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_005.txt,berhasil_list_html,342
5,case_006,1527/Pid.B/2014/PN.TNG,16 September 2014,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_006.txt,berhasil_list_html,276
6,case_007,834 / Pid.B / 2017 / PN.TNG.,15 Juni 2017,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_007.txt,berhasil_list_html,263
7,case_008,2000/Pid.B/2017/PN.Tng,16 Nopember 2017,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_008.txt,berhasil_list_html,349
8,case_009,2497/Pid.B/2018/PN Tng,10 Januari 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_009.txt,berhasil_list_html,73
9,case_010,329/Pid.B/2019/PN Tng,1 April 2019,PN Tangerang,Pidana Umum - Pencurian,https://putusan3.mahkamahagung.go.id/direktori...,case_010.txt,berhasil_list_html,335
